# Employee Attrition Prediction - Dashboard Outputs

This notebook creates dashboard-ready summary tables for Power BI. It reads the prepared employee attrition table and model metrics table from Databricks, then generates business summary outputs for retention monitoring and stakeholder reporting.

# 1. Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F

# 2. Load Reusable Databricks Tables

In [0]:
employee_df = spark.table("attrition_project.employee_attrition")
model_metrics_df = spark.table("attrition_project.model_metrics")

print(f"Employee rows: {employee_df.count()}")
print(f"Employee columns: {len(employee_df.columns)}")

display(employee_df.limit(5))
display(model_metrics_df)

Employee rows: 1470
Employee columns: 35


Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


model,accuracy,attrition_precision,attrition_recall,attrition_f1,roc_auc
logistic_regression,0.764172335600907,0.37209302325581395,0.676056338028169,0.48,0.8162923486867149
random_forest,0.8299319727891157,0.46,0.323943661971831,0.38016528925619836,0.7738104301484583
gradient_boosting,0.8435374149659864,0.53125,0.23943661971830985,0.3300970873786408,0.7835173201370385


# 3. Clean Data

In [0]:
model_metrics_dashboard = model_metrics_df.select(
    "model",
    F.round(F.col("accuracy") * 100, 2).alias("accuracy_pct"),
    F.round(F.col("attrition_precision") * 100, 2).alias("attrition_precision_pct"),
    F.round(F.col("attrition_recall") * 100, 2).alias("attrition_recall_pct"),
    F.round(F.col("attrition_f1") * 100, 2).alias("attrition_f1_pct"),
    F.round(F.col("roc_auc") * 100, 2).alias("roc_auc_pct"),
)

display(model_metrics_dashboard)

model,accuracy_pct,attrition_precision_pct,attrition_recall_pct,attrition_f1_pct,roc_auc_pct
logistic_regression,76.42,37.21,67.61,48.0,81.63
random_forest,82.99,46.0,32.39,38.02,77.38
gradient_boosting,84.35,53.13,23.94,33.01,78.35


##Save Model Metrics Dashboard

In [0]:
model_metrics_dashboard.write.mode("overwrite").saveAsTable(
    "attrition_project.dashboard_model_metrics"
)

# 4. Create Attrition Overview Table

This table provides high-level KPI metrics for the Power BI dashboard, including total employees, attrition count, retained count, and overall attrition rate.

In [0]:
attrition_overview = employee_df.agg(
    F.count("*").alias("total_employees"),
    F.sum(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)).alias("attrition_count"),
    F.sum(F.when(F.col("Attrition") == "No", 1).otherwise(0)).alias("retained_count"),
    F.round(
        100.0 * F.avg(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)),
        2
    ).alias("attrition_rate_pct")
)

display(attrition_overview)

total_employees,attrition_count,retained_count,attrition_rate_pct
1470,237,1233,16.12


## Save Attrition Overview Table

In [0]:
attrition_overview.write.mode("overwrite").saveAsTable(
    "attrition_project.dashboard_attrition_overview"
)

# 5. Create Attrition by Overtime Table

In [0]:
attrition_by_overtime = (
    employee_df
    .groupBy("OverTime")
    .agg(
        F.count("*").alias("employee_count"),
        F.sum(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)).alias("attrition_count"),
        F.round(
            100.0 * F.avg(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)),
            2
        ).alias("attrition_rate_pct")
    )
    .orderBy(F.col("attrition_rate_pct").desc())
)

display(attrition_by_overtime)

OverTime,employee_count,attrition_count,attrition_rate_pct
Yes,416,127,30.53
No,1054,110,10.44


## Save Attrition by Overtime Table

In [0]:
attrition_by_overtime.write.mode("overwrite").saveAsTable(
    "attrition_project.dashboard_attrition_by_overtime"
)

## 6. Create Attrition by Department and Tenure Table

In [0]:
attrition_by_department_tenure = (
    employee_df
    .withColumn(
        "tenure_band",
        F.when(F.col("YearsAtCompany") < 1, "<1 year")
         .when(F.col("YearsAtCompany").between(1, 2), "1-2 years")
         .when(F.col("YearsAtCompany").between(3, 5), "3-5 years")
         .otherwise("6+ years")
    )
    .groupBy("Department", "tenure_band")
    .agg(
        F.count("*").alias("employee_count"),
        F.sum(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)).alias("attrition_count"),
        F.round(
            100.0 * F.avg(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)),
            2
        ).alias("attrition_rate_pct")
    )
    .orderBy(F.col("attrition_rate_pct").desc(), F.col("employee_count").desc())
)

display(attrition_by_department_tenure)

Department,tenure_band,employee_count,attrition_count,attrition_rate_pct
Human Resources,1-2 years,13,6,46.15
Sales,<1 year,17,7,41.18
Sales,1-2 years,86,29,33.72
Research & Development,<1 year,27,9,33.33
Research & Development,1-2 years,199,51,25.63
Sales,3-5 years,117,24,20.51
Human Resources,3-5 years,23,4,17.39
Sales,6+ years,226,32,14.16
Research & Development,3-5 years,294,32,10.88
Research & Development,6+ years,441,41,9.3


## Save Attrition by Department and Tenure Table

In [0]:
attrition_by_department_tenure.write.mode("overwrite").saveAsTable(
    "attrition_project.dashboard_attrition_by_department_tenure"
)

## 7. Create Attrition by Travel and Commute Table

In [0]:
attrition_by_travel_commute = (
    employee_df
    .withColumn(
        "BusinessTravel",
        F.regexp_replace(F.regexp_replace(F.col("BusinessTravel"), "_", " "), "-", " ")
    )
    .withColumn(
        "commute_band",
        F.when(F.col("DistanceFromHome") <= 5, "0-5 miles")
         .when(F.col("DistanceFromHome") <= 15, "6-15 miles")
         .otherwise("16+ miles")
    )
    .withColumn(
        "commute_sort",
        F.when(F.col("DistanceFromHome") <= 5, 1)
         .when(F.col("DistanceFromHome") <= 15, 2)
         .otherwise(3)
    )
    .groupBy("BusinessTravel", "commute_band", "commute_sort")
    .agg(
        F.count("*").alias("employee_count"),
        F.sum(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)).alias("attrition_count"),
        F.round(F.avg("JobSatisfaction"), 2).alias("avg_job_satisfaction"),
        F.round(
            100.0 * F.avg(F.when(F.col("Attrition") == "Yes", 1).otherwise(0)),
            2
        ).alias("attrition_rate_pct")
    )
    .filter(F.col("employee_count") >= 10)
    .orderBy("commute_sort")
)

display(attrition_by_travel_commute)

BusinessTravel,commute_band,commute_sort,employee_count,attrition_count,avg_job_satisfaction,attrition_rate_pct
Travel Frequently,0-5 miles,1,125,28,2.75,22.4
Non Travel,0-5 miles,1,63,2,2.78,3.17
Travel Rarely,0-5 miles,1,444,57,2.75,12.84
Travel Rarely,6-15 miles,2,375,57,2.66,15.2
Non Travel,6-15 miles,2,48,4,2.9,8.33
Travel Frequently,6-15 miles,2,86,21,2.73,24.42
Non Travel,16+ miles,3,39,6,2.69,15.38
Travel Frequently,16+ miles,3,66,20,2.94,30.3
Travel Rarely,16+ miles,3,224,42,2.68,18.75


## Save Attrition by Travel and Commute Table

In [0]:
attrition_by_travel_commute.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("attrition_project.dashboard_attrition_by_travel_commute")

## 8. Dashboard Tables Summary

In [0]:
dashboard_tables = [
    "attrition_project.dashboard_model_metrics",
    "attrition_project.dashboard_attrition_overview",
    "attrition_project.dashboard_attrition_by_overtime",
    "attrition_project.dashboard_attrition_by_department_tenure",
    "attrition_project.dashboard_attrition_by_travel_commute",
]

for table_name in dashboard_tables:
    table_df = spark.table(table_name)
    print(f"{table_name}: {table_df.count()} rows")

attrition_project.dashboard_model_metrics: 3 rows
attrition_project.dashboard_attrition_overview: 1 rows
attrition_project.dashboard_attrition_by_overtime: 2 rows
attrition_project.dashboard_attrition_by_department_tenure: 11 rows
attrition_project.dashboard_attrition_by_travel_commute: 9 rows
